# Volatility Targeting Avancé — Position Cap & Hystérésis

Extension de `vol_targeting.ipynb` avec deux améliorations :

1. **Cap de position** : limite le levier maximum (`|pos| ≤ pos_cap`)
2. **Hystérésis CMM** : filtre les faux croisements via un seuil δ (trigger de Schmitt)

Tests statistiques de significativité (Diebold-Mariano, Mann-Whitney).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import norm, mannwhitneyu

from Backtester.Backtester import compute_backtest, plot_backtest
from prediction.settings import PAIRS, N_TARGETS, DATA_DIR, OUTPUT_DIR, CONTEXT_LENGTH

TC_BPS        = 5.0
FAST_MA_GRID  = [4, 8, 16, 32]
SLOW_MA_GRID  = [48, 96, 192, 384]
EWMA_SPAN_GRID = [4, 8, 16]
DELTA_GRID    = [0.0, 0.0005, 0.001, 0.002, 0.005, 0.01]
POS_CAP       = 5.0   # levier max |pos| ≤ 5×

print(f'Imports OK  |  Paires : {PAIRS}')

In [ ]:
def compute_signal_hysteresis(prices_df, fast, slow, delta=0.0):
    '''Signal CMM avec hystérésis (trigger de Schmitt).
    delta=0 → CMM standard. delta>0 → retournement seulement si |gap| > delta.
    '''
    ma_fast = prices_df.rolling(fast, min_periods=1).mean()
    ma_slow = prices_df.rolling(slow, min_periods=1).mean()
    if delta == 0.0:
        return pd.DataFrame(
            np.where(ma_fast.values > ma_slow.values, 1., -1.),
            index=prices_df.index, columns=prices_df.columns
        )
    gap = (ma_fast - ma_slow) / (ma_slow.replace(0, np.nan))
    trans = pd.DataFrame(np.nan, index=prices_df.index, columns=prices_df.columns)
    trans[gap >  delta] =  1.0
    trans[gap < -delta] = -1.0
    return trans.ffill().fillna(1.0)


def build_positions(preds_df, prices_df, fast, slow, ewma_span, vol_cible, delta=0.0):
    '''Signal (avec hystérésis optionnelle) × (vol_cible / vol_lissée).'''
    sig       = compute_signal_hysteresis(prices_df, fast, slow, delta)
    vol_lis   = preds_df.abs().ewm(span=ewma_span, adjust=False).mean().clip(lower=1e-6)
    cidx      = sig.index.intersection(vol_lis.index)
    pos       = sig.loc[cidx] * (vol_cible / vol_lis.loc[cidx])
    return pos.clip(-10, 10).reindex(prices_df.index, fill_value=0)


def build_positions_with_cap(preds_df, prices_df, fast, slow, ewma_span, vol_cible,
                              delta=0.0, pos_cap=POS_CAP):
    '''Vol targeting avec cap absolu sur |position|.'''
    pos = build_positions(preds_df, prices_df, fast, slow, ewma_span, vol_cible, delta)
    return pos.clip(-pos_cap, pos_cap)


def get_metric(bt_obj, key, row=None):
    if bt_obj is None: return np.nan
    df   = bt_obj.metrics.toDataFrame()
    cols = [c for c in df.columns if key.lower() in c.lower()]
    if not cols: return np.nan
    r = row if (row and row in df.index) else df.index[0]
    try:
        return float(str(df.loc[r, cols[0]]).replace('%','').replace('bps','').strip())
    except Exception:
        return np.nan


def sharpe_fast(pos_df, perf_df, tc_bps=TC_BPS):
    common = pos_df.index.intersection(perf_df.index)
    bt = compute_backtest(
        pos_df.loc[common], perf_df.loc[common],
        transaction_costs_bps=tc_bps, compute_max_DD=False, show_pnl=False,
        position_lag_value=1, check_index=True, position_fill_value=0.0
    )
    row = 'Total' if (pos_df.ndim > 1 and pos_df.shape[1] > 1) else None
    return get_metric(bt, 'sharpe', row=row)


def extract_pnl_daily(pos_df, ret_df):
    common = pos_df.index.intersection(ret_df.index)
    pnl = (pos_df.loc[common] * ret_df.loc[common]).sum(axis=1)
    daily = pnl.resample('1D').sum().dropna()
    return daily[daily.index.dayofweek < 7]  # crypto 7j/7


def extract_trade_pnl(pos_df, ret_df, pair):
    common   = pos_df.index.intersection(ret_df.index)
    pos_arr  = pos_df.loc[common, pair].values
    pnl_arr  = (pos_df.loc[common, pair] * ret_df.loc[common, pair]).values
    sign_v   = np.sign(pos_arr)
    changes  = np.where(np.diff(sign_v, prepend=sign_v[0]) != 0)[0]
    bounds   = np.concatenate([[0], changes, [len(pos_arr)]])
    trades   = []
    for i in range(len(bounds) - 1):
        s, e = bounds[i], bounds[i+1]
        if np.any(pos_arr[s:e] != 0):
            trades.append(pnl_arr[s:e].sum())
    return np.array(trades)


def plot_pnl(pos_df, ret_df, title, val_s=None, test_s=None):
    common = pos_df.index.intersection(ret_df.index)
    pnl = (pos_df.loc[common] * ret_df.loc[common]).sum(axis=1).cumsum()
    fig, ax = plt.subplots(figsize=(14, 4))
    pnl.plot(ax=ax, color='steelblue', lw=0.8)
    if val_s  is not None: ax.axvline(val_s,  color='green',  ls='--', lw=1.2, label='Validation')
    if test_s is not None: ax.axvline(test_s, color='orange', ls='--', lw=1.2, label='Test')
    ax.axhline(0, color='black', lw=0.5, ls=':')
    ax.set_title(title); ax.set_ylabel('PnL cumulé')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


print('Fonctions utilitaires OK')

In [ ]:
# ── Chargement des prédictions ────────────────────────────────────────
preds = {}
for split in ('train', 'validation', 'test'):
    df = pd.read_parquet(OUTPUT_DIR / f'{split}_lstm_1l_predictions.parquet')
    df.columns = PAIRS
    preds[split] = df

train_preds = preds['train']
val_preds   = preds['validation']
test_preds  = preds['test']
all_preds   = pd.concat(list(preds.values())).sort_index()

train_start, train_end = train_preds.index[0],  train_preds.index[-1]
val_start,   val_end   = val_preds.index[0],    val_preds.index[-1]
test_start,  test_end  = test_preds.index[0],   test_preds.index[-1]

# ── Prix et rendements 15-min ─────────────────────────────────────────
prices_list, returns_list = [], []
for crypto in PAIRS:
    df1m  = pd.read_parquet(DATA_DIR / f'{crypto}_1m.parquet')
    close = df1m.loc[df1m.index.minute % 15 == 0, 'close'].sort_index()
    ret   = np.log(close / close.shift(1))
    prices_list.append(close.rename(crypto))
    returns_list.append(ret.rename(crypto))

prices_df  = pd.concat(prices_list,  axis=1).sort_index()
returns_df = pd.concat(returns_list, axis=1).sort_index()

vol_med  = all_preds.loc[train_start:train_end].abs().median()
vol_q75  = all_preds.loc[train_start:train_end].abs().quantile(0.75)
vc_opts  = {'median': vol_med, 'q75': vol_q75}

print(f'Train : {train_start.date()} → {train_end.date()}')
print(f'Valid : {val_start.date()}   → {val_end.date()}')
print(f'Test  : {test_start.date()}  → {test_end.date()}')
print('Données chargées OK')

In [ ]:
# ── Grid Search de base (sans hystérésis) — reproduit vol_targeting.ipynb
val_returns_g = returns_df.loc[val_start:val_end]
best_sharpe, best_params = -999., {}
results = []

for fast in FAST_MA_GRID:
    for slow in SLOW_MA_GRID:
        if fast >= slow: continue
        for ewma in EWMA_SPAN_GRID:
            for vc_key, vc in vc_opts.items():
                vpos = build_positions(all_preds, prices_df, fast, slow, ewma, vc, delta=0.0)
                sh   = sharpe_fast(vpos.loc[val_start:val_end], val_returns_g)
                results.append({'fast': fast, 'slow': slow, 'ewma': ewma, 'vol': vc_key, 'sharpe': round(sh, 4)})
                if sh > best_sharpe:
                    best_sharpe = sh
                    best_params = {'fast': fast, 'slow': slow, 'ewma': ewma, 'vol': vc_key}

vc_best = vol_q75 if best_params['vol'] == 'q75' else vol_med
pos_base = build_positions(
    all_preds, prices_df,
    best_params['fast'], best_params['slow'], best_params['ewma'], vc_best
)
bt_base_test = compute_backtest(
    pos_base.loc[test_start:test_end], returns_df.loc[test_start:test_end],
    detail_assets=True, position_lag_value=1, check_index=True, position_fill_value=0.0,
    compute_max_DD=True, transaction_costs_bps=TC_BPS
)
print(f'Best params (sans hystérésis) : {best_params}  (Sharpe val = {best_sharpe:.3f})')
print(bt_base_test.metrics.toDataFrame().to_string())

In [ ]:
# ── Phase 1 : Remède — Cap de position (|pos| ≤ POS_CAP) ──────────────
pos_cap = build_positions_with_cap(
    all_preds, prices_df,
    best_params['fast'], best_params['slow'], best_params['ewma'], vc_best,
    pos_cap=POS_CAP
)

# Distribution des tailles de position : base vs cap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, pos, lbl in [(axes[0], pos_base, 'Sans cap'), (axes[1], pos_cap, f'Cap {POS_CAP}×')]:
    for crypto, color in zip(PAIRS, ['steelblue', 'darkorange']):
        pos[crypto].abs().loc[test_start:test_end].plot(
            ax=ax, color=color, lw=0.5, alpha=0.7, label=crypto)
    ax.axhline(POS_CAP, color='red', ls='--', lw=1, label=f'Cap {POS_CAP}×')
    ax.set_title(f'|Position| — {lbl} (test)'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

bt_cap_test = compute_backtest(
    pos_cap.loc[test_start:test_end], returns_df.loc[test_start:test_end],
    detail_assets=True, position_lag_value=1, check_index=True, position_fill_value=0.0,
    compute_max_DD=True, transaction_costs_bps=TC_BPS
)
print(f'── Cap {POS_CAP}× (test) ────────────────────────────────────────')
print(bt_cap_test.metrics.toDataFrame().to_string())

In [ ]:
# ── Phase 2 : Grid Search avec hystérésis ─────────────────────────────
best_sh_hyst, best_params_hyst = -999., {}
results_hyst = []

n_total = sum(1 for f in FAST_MA_GRID for s in SLOW_MA_GRID if f < s) * len(EWMA_SPAN_GRID) * 2 * len(DELTA_GRID)
n_done  = 0
for fast in FAST_MA_GRID:
    for slow in SLOW_MA_GRID:
        if fast >= slow: continue
        for ewma in EWMA_SPAN_GRID:
            for vc_key, vc in vc_opts.items():
                for delta in DELTA_GRID:
                    vpos = build_positions(all_preds, prices_df, fast, slow, ewma, vc, delta=delta)
                    sh   = sharpe_fast(vpos.loc[val_start:val_end], val_returns_g)
                    results_hyst.append({'fast': fast, 'slow': slow, 'ewma': ewma,
                                         'vol': vc_key, 'delta': delta, 'sharpe': round(sh, 4)})
                    if sh > best_sh_hyst:
                        best_sh_hyst   = sh
                        best_params_hyst = {'fast': fast, 'slow': slow, 'ewma': ewma,
                                            'vol': vc_key, 'delta': delta}
                    n_done += 1
print(f'Grid search hystérésis terminée ({n_done} combinaisons)')

df_hyst = pd.DataFrame(results_hyst).sort_values('sharpe', ascending=False)
print('\nTOP 10 (avec hystérésis) :')
print(df_hyst.head(10).to_string(index=False))
print(f'\nMeilleurs params (avec hystérésis) : {best_params_hyst}  (Sharpe = {best_sh_hyst:.3f})')

In [ ]:
# ── Phase 3 : Comparaison sans/avec hystérésis (test) ─────────────────
vc_hyst = vol_q75 if best_params_hyst['vol'] == 'q75' else vol_med
pos_hyst = build_positions(
    all_preds, prices_df,
    best_params_hyst['fast'], best_params_hyst['slow'], best_params_hyst['ewma'],
    vc_hyst, delta=best_params_hyst['delta']
)

bt_hyst_test = compute_backtest(
    pos_hyst.loc[test_start:test_end], returns_df.loc[test_start:test_end],
    detail_assets=True, position_lag_value=1, check_index=True, position_fill_value=0.0,
    compute_max_DD=True, transaction_costs_bps=TC_BPS
)
print(f'── Avec hystérésis δ={best_params_hyst["delta"]} (test) ─────────────────────')
print(bt_hyst_test.metrics.toDataFrame().to_string())

# PnL cumulé comparatif
pair = PAIRS[0]  # BTC
fig, ax = plt.subplots(figsize=(14, 4))
for pos, lbl, color in [
    (pos_base,  'Sans hystérésis',  'steelblue'),
    (pos_hyst,  f'δ={best_params_hyst["delta"]}', 'darkorange'),
    (pos_cap,   f'Cap {POS_CAP}×', 'green'),
]:
    common_t = pos.index.intersection(returns_df.loc[test_start:test_end].index)
    pnl = (pos.loc[common_t, pair] * returns_df.loc[common_t, pair]).cumsum()
    pnl.plot(ax=ax, label=lbl, lw=1.0, color=color)
ax.set_title(f'{pair} — PnL cumulé (test) : comparaison stratégies')
ax.axhline(0, color='black', lw=0.5, ls=':')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Phase 4 : Tests Statistiques ──────────────────────────────────────
def newey_west_var(d, bw):
    n = len(d); gamma0 = np.var(d, ddof=1); nw = gamma0
    for lag in range(1, bw + 1):
        nw += 2 * (1 - lag/(bw+1)) * np.cov(d[lag:], d[:-lag])[0,1]
    return max(nw, 1e-16) / n

def dm_test(pnl1, pnl2):
    common = pnl1.index.intersection(pnl2.index)
    d = (pnl1.loc[common] - pnl2.loc[common]).dropna().values
    n = len(d)
    if n < 10: return np.nan, np.nan, n
    bw = math.ceil(n ** (1/3))
    v  = newey_west_var(d, bw)
    dm = np.mean(d) / np.sqrt(v)
    pv = 2 * (1 - norm.cdf(abs(dm)))
    return dm, pv, n

# PnL journaliers (test)
pnl_base = extract_pnl_daily(pos_base.loc[test_start:test_end],  returns_df.loc[test_start:test_end])
pnl_hyst = extract_pnl_daily(pos_hyst.loc[test_start:test_end],  returns_df.loc[test_start:test_end])
pnl_cap  = extract_pnl_daily(pos_cap.loc[test_start:test_end],   returns_df.loc[test_start:test_end])

print('=== Tests de Diebold-Mariano (PnL journaliers — test) ===')
print(f'{"Comparaison":<35} {"DM":>8} {"p-val":>8} {"n":>6} {"Sig5%":>6}')
print('-' * 65)
for lbl, p1, p2 in [
    (f'Hystérésis vs Base',   pnl_hyst, pnl_base),
    (f'Cap {POS_CAP}× vs Base',  pnl_cap,  pnl_base),
    (f'Hystérésis vs Cap',    pnl_hyst, pnl_cap),
]:
    dm, pv, n = dm_test(p1, p2)
    sig = 'Oui' if (not np.isnan(pv) and pv < 0.05) else 'Non'
    dm_s = f'{dm:+.3f}' if not np.isnan(dm) else 'NaN'
    pv_s = f'{pv:.3f}'  if not np.isnan(pv) else '---'
    print(f'{lbl:<35} {dm_s:>8} {pv_s:>8} {n:>6} {sig:>6}')

# Mann-Whitney sur PnL par trade (BTC)
print('\n=== Mann-Whitney PnL/trade (BTC, test) ===')
t_base = extract_trade_pnl(pos_base.loc[test_start:test_end], returns_df.loc[test_start:test_end], PAIRS[0])
t_hyst = extract_trade_pnl(pos_hyst.loc[test_start:test_end], returns_df.loc[test_start:test_end], PAIRS[0])
for name, t in [('Base', t_base), ('Hystérésis', t_hyst)]:
    print(f'{name:12s}: {len(t):4d} trades  médiane={np.median(t)*1e4:+.2f} bps')
if len(t_base) > 1 and len(t_hyst) > 1:
    stat, pv = mannwhitneyu(t_base, t_hyst, alternative='two-sided')
    print(f'U={stat:.0f}  p={pv:.4f}  Sig5%={"Oui" if pv < 0.05 else "Non"}')

In [ ]:
# ── Phase 5 : Tableau de Synthèse ─────────────────────────────────────
METRICS = [('Sharpe','sharpe'), ('Return/an','return (yearly)'), ('PnL/Trade','PnL/Trade'), ('Max DD','max DD')]
rows = []
for cfg, bt in [
    ('Base (best params)',             bt_base_test),
    (f'Cap |pos|≤{POS_CAP}',          bt_cap_test),
    (f'Hystérésis δ={best_params_hyst["delta"]}', bt_hyst_test),
]:
    for asset in PAIRS + ['Total']:
        r = {'Config': cfg, 'Actif': asset}
        for lbl, key in METRICS:
            r[lbl] = get_metric(bt, key, row=asset)
        rows.append(r)

summary = pd.DataFrame(rows).set_index(['Config', 'Actif'])
print('=' * 70)
print('  SYNTHÈSE — Performances sur le jeu de test (net frais)')
print('=' * 70)
print(summary.to_string())